In [1]:
import sys
import glob
import warnings
import numpy as np
import pandas as pd
# import seaborn as sns
# import datetime as dt
# import matplotlib.cm as cm
import matplotlib.pyplot as plt
# import logging

# from math import sqrt
from prophet import Prophet
from pmdarima import auto_arima
from xgboost import XGBRegressor
from pathlib import Path
sys.path.append(str(Path.cwd().parent.parent))

# from src.utils import util as utl
from itertools import product
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

global companies_set 

warnings.filterwarnings('ignore')

c:\Users\Caio Medeiros\Documents\caio_pessoal\pos-graduacao_gran_curso\TCC\analise_de_indicador_tcc_ufcg\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1 - Estudo das medidas de tendências centrais

In [3]:
markets = ['AN', 'AS', 'EU', 'JP', 'BRL']
df = pd.read_csv('../../data/grouped_markets/grouped_markets_time_series.csv')

statistics = []

for market in markets:
    time_serie = df[market]

    statistics.append({
        "Mercado": market,
        "Média": time_serie.mean(),
        "Mediana": time_serie.median(),
        "Desvio padrão": time_serie.std(),
        "Variância": time_serie.var(),
        "Mínimo": time_serie.min(),
        "Máximo": time_serie.max(),
        "Amplitude": time_serie.max() - time_serie.min(),
        "Coeficiente de variação (%)": (time_serie.std()/time_serie.mean())*100
    })


In [ ]:
# Maior média:
statistics = pd.DataFrame(statistics).sort_values(by='Média', ascending=False)
print("Maior média: ")
display(statistics.round(2).iloc[[0]])

# Maior Mediana:
statistics = pd.DataFrame(statistics).sort_values(by='Mediana', ascending=False)
print("Maior mediana: ")
display(statistics.round(2).iloc[[0]])

# Maior Desvio padrão:
statistics = pd.DataFrame(statistics).sort_values(by='Desvio padrão', ascending=False)
print("Maior desvio padrão: ")
display(statistics.round(2).iloc[[0]])

# Maior Variância:
statistics = pd.DataFrame(statistics).sort_values(by='Variância', ascending=False)
print("Maior variância: ")
display(statistics.round(2).iloc[[0]])

# Maior Máximo:
statistics = pd.DataFrame(statistics).sort_values(by='Máximo', ascending=False)
print("Maior máximo: ")
display(statistics.round(2).iloc[[0]])

# Menor Mínimo:
statistics = pd.DataFrame(statistics).sort_values(by='Mínimo', ascending=True)
print("Menor mínimo: ")
display(statistics.round(2).iloc[[0]])


Maior média: 


,Mercado,Média,Mediana,Desvio padrão,Variância,Mínimo,Máximo,Amplitude,Coeficiente de variação (%)
2,EU,78186.22,77582.0,13762.45,1.894051e+08,36852.0,108902.0,72050.0,17.6


Maior mediana: 


,Mercado,Média,Mediana,Desvio padrão,Variância,Mínimo,Máximo,Amplitude,Coeficiente de variação (%)
2,EU,78186.22,77582.0,13762.45,1.894051e+08,36852.0,108902.0,72050.0,17.6


Maior desvio padrão: 


...

,Mercado,Média,Mediana,Desvio padrão,Variância,Mínimo,Máximo,Amplitude,Coeficiente de variação (%)
1,AS,55059.22,53200.0,25169.68,6.335128e+08,9450.0,125724.0,116274.0,45.71


Maior máximo: 


,Mercado,Média,Mediana,Desvio padrão,Variância,Mínimo,Máximo,Amplitude,Coeficiente de variação (%)
1,AS,55059.22,53200.0,25169.68,6.335128e+08,9450.0,125724.0,116274.0,45.71


Menor mínimo: 


,Mercado,Média,Mediana,Desvio padrão,Variância,Mínimo,Máximo,Amplitude,Coeficiente de variação (%)
3,JP,36450.12,37093.0,17874.55,3.194996e+08,0.0,98688.0,98688.0,49.04


In [24]:
# Pegando a razão entre maior média e menor média:
maior_media = statistics.iloc[0]["Média"]
menor_media = statistics.iloc[-1]["Média"]   # última linha

ratio = np.round(maior_media / menor_media, 2)

print("Razão entre maior e menor média dos mercados:", ratio)

Razão entre maior e menor média dos mercados: 22.05
